In [2]:
print("XXX")

XXX


In [3]:
from email.header import decode_header

def decode_mime_header(s):
    """エンコードされたヘッダー（ファイル名など）をデコードする"""
    if not s:
        return ""
    parts = decode_header(s)
    decoded_parts = []
    for payload, charset in parts:
        if isinstance(payload, bytes):
            decoded_parts.append(payload.decode(charset or "utf-8", errors="replace"))
        else:
            decoded_parts.append(payload)
    return "".join(decoded_parts)


In [4]:
from email.header import decode_header, make_header
from email.mime import message
from email.utils import getaddresses

def safe_decode_header(s):
    """構造を維持したままMIMEデコードする"""
    if not s:
        return ""
    try:
        # decode_header + make_header はRFCに基づき構造を維持してデコードする
        return str(make_header(decode_header(s)))
    except Exception:
        return s



In [5]:
import re

def extract_name_email(s):
    # 文末にある <...> を探す正規表現
    # (.*)  : 任意文字（name候補）
    # \s*   : 空白（あれば）
    # <([^>]+)> : < > で囲まれた中身をキャプチャ
    # $     : 行末
    match = re.search(r'^(.*)\s*<([^>]+)>$', s.strip())
    
    if match:
        name = match.group(1).strip()
        email = match.group(2).strip()
        
        # もし name が " や < > で囲まれていたら剥ぎ取る
        name = name.strip('"').strip('<').strip('>')
        return name, email
    else:
        return s, ""

# name, email = extract_name_email(A)
# print(f"name : {name}")
# print(f"email: {email}")

In [6]:
import mailbox
import csv
from email.utils import getaddresses, parsedate_to_datetime
# from email.mime import message

def extract_attachments_list(mbox_path, output_csv):
    mbox = mailbox.mbox(mbox_path, factory=None)
    
    # データを一時的に格納するリスト
    data_list = []

    for message in mbox:
        try:
            # 日付の取得とパース
            dt = parsedate_to_datetime(message['Date']) if message['Date'] else None
            # ソートのために、日付は文字列ではなく datetime オブジェクトのまま保持するか、
            # 文字列にするなら '2023/01/01' 形式（比較可能）にします
            date_str = dt.strftime('%Y/%m/%d') if dt else "0000/00/00"

            # # 差出人情報の取得
            # raw_sender = decode_mime_header(message['From']) if message['From'] else ""
            # if raw_sender:
            #     name, email = getaddresses([raw_sender])[0]
            # else:
            #     name, email = "", ""


            # --- メインの処理 ---
            raw_from = message.get('From', '')

            if raw_from:
                # 1. 構造を壊さずにデコード（重要！）
                # ここで元の関数の join() を使うのではなく、make_header に任せる
                decoded_sender = safe_decode_header(raw_from)
                
                # 2. パース
                parsed = getaddresses([decoded_sender])
                
                if parsed:
                    name, email = parsed[0]
                    # name が " で囲まれたままなら strip する
                    name = name.strip('"')
                    if name=="" : 
                        name, email = extract_name_email(decoded_sender)
                    if name=="" : 
                        name=decoded_sender
    
                else:
                    name, email = "", ""
            else:
                name, email = "", ""

            if email == "" and len(name) > 0 : email = name
            if name  == "" and len(email) > 0 : name = email
            
            # subject = decode_mime_header(message['Subject']) if message['Subject'] else ""

            # --- Subjectの改行対策 ---
            raw_subject = decode_mime_header(message['Subject']) if message['Subject'] else ""
            # 改行をスペースに置換して1行にする
            subject = raw_subject.replace('\n', ' ').replace('\r', ' ').strip()




            # リストに追加（この時点では順不同）
            data_list.append({
                'date': date_str,
                'name': name,
                'email': email,
                'subject': subject
            })

        except Exception as e:
            print(f"Skipping a message due to error: {e}")
            continue

    # --- ソート処理 ---
    # lambdaを使って (メールアドレス, 日付) の優先順位でソートします
    data_list.sort(key=lambda x: (x['email'], x['date']))

    # --- CSV書き出し ---
    with open(output_csv, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(['Date', 'From(Name)', 'From(Email)', 'Subject'])
        
        for item in data_list:
            writer.writerow([item['date'], item['name'], item['email'], item['subject']])

    print(f"完了！ {output_csv} を確認してください。")

In [7]:
import os
import glob

def process_all_mboxes(input_dir, output_dir):
    # 1) output_y フォルダが存在しない場合は作成
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created directory: {output_dir}")

    # 3) 指定ディレクトリ配下の .mbox ファイルをすべて取得
    # 4) 4桁の年.mbox (例: 2020.mbox) にマッチするものを探す
    mbox_files = glob.glob(os.path.join(input_dir, "[0-9][0-9][0-9][0-9].mbox"))

    if not mbox_files:
        print("No matching .mbox files found.")
        return

    for file_path in mbox_files:
        # ファイル名（2020.mboxなど）を取得
        base_name = os.path.basename(file_path)
        
        # 2) csvファイル名を設定（例: 2020.mbox.csv または 2020.csv）
        # ここでは 2020.mbox.csv となるように設定しています
        # csv_filename = f"{base_name}.csv"
        csv_filename = base_name.replace('.mbox', '.csv')
        output_path = os.path.join(output_dir, csv_filename)

        print(f"Processing: {base_name} -> {output_path}")
        
        # 既存の関数を実行
        try:
            extract_attachments_list(file_path, output_path)

        # メールのヘッダーや本文に charset="unknown-8bit" という記述がある場合、
        # Pythonの標準ライブラリ（email や mailbox）は「そんな名前の文字コードは
        # 知らない」とエラーを返します。これは古いメールサーバーや、特定の
        # スパムメール、あるいは文字化けしたメールでよく見られる現象です。

        except Exception as e:
            print(f"Error processing {base_name}: {e}")

        # break

# --- 実行セクション ---
input_directory = "/home/yutaka/src_p/260506_Google_mail/split_mbox_files"
output_directory = "/home/yutaka/src_p/260510_Google_mail/output_y"
process_all_mboxes(input_directory, output_directory)

Processing: 2021.mbox -> /home/yutaka/src_p/260510_Google_mail/output_y/2021.csv
完了！ /home/yutaka/src_p/260510_Google_mail/output_y/2021.csv を確認してください。
Processing: 2019.mbox -> /home/yutaka/src_p/260510_Google_mail/output_y/2019.csv
完了！ /home/yutaka/src_p/260510_Google_mail/output_y/2019.csv を確認してください。
Processing: 2022.mbox -> /home/yutaka/src_p/260510_Google_mail/output_y/2022.csv
完了！ /home/yutaka/src_p/260510_Google_mail/output_y/2022.csv を確認してください。
Processing: 2026.mbox -> /home/yutaka/src_p/260510_Google_mail/output_y/2026.csv
完了！ /home/yutaka/src_p/260510_Google_mail/output_y/2026.csv を確認してください。
Processing: 2024.mbox -> /home/yutaka/src_p/260510_Google_mail/output_y/2024.csv
完了！ /home/yutaka/src_p/260510_Google_mail/output_y/2024.csv を確認してください。
Processing: 2016.mbox -> /home/yutaka/src_p/260510_Google_mail/output_y/2016.csv
Skipping a message due to error: unknown encoding: unknown-8bit
完了！ /home/yutaka/src_p/260510_Google_mail/output_y/2016.csv を確認してください。
Processing: 2023.mbox ->